# Emerging Technologies
### Nathan Carr - G00410214

## Problem 1: Generating Random Boolean Functions

The Deutsch–Jozsa algorithm is designed to work with functions that accept a fixed number of Boolean inputs and return a single Boolean output. Each function is guaranteed to be either constant (always returns False or always returns True) or balanced (returns True for exactly half of the possible input combinations). Write a Python function random_constant_balanced that returns a randomly chosen function from the set of constant or balanced functions taking four Boolean arguments as inputs.

A 4-bit Boolean function has 16 possible inputs. I generate a random promised function by building a truth table over all inputs. If the function is constant, every entry is the same Boolean value. If it is balanced, I randomly select exactly 8 of the 16 inputs to map to `True` (the rest map to `False`). The function returned is a callable that looks up its output from this table.

In [1]:
import random
from itertools import product

def random_constant_balanced(seed: int | None = None):
    """
    Return a random 4-input Boolean function that is either constant or balanced.

    Constant: always False or always True.
    Balanced: True for exactly 8 of the 16 possible inputs.
    """
    rng = random.Random(seed)
    inputs = list(product([False, True], repeat=4))  # 16 input tuples

    if rng.choice([True, False]):  # balanced
        true_inputs = set(rng.sample(inputs, k=8))   # choose exactly half
        table = {x: (x in true_inputs) for x in inputs}
    else:  # constant
        const_value = rng.choice([False, True])
        table = {x: const_value for x in inputs}

    def f(a: bool, b: bool, c: bool, d: bool) -> bool:
        return table[(a, b, c, d)]

    # Helpful for demonstration/testing in your notebook
    f.truth_table = table  # type: ignore[attr-defined]

    return f


In [2]:
# Generate a few functions and verify they match the promise
for s in range(5):
    f = random_constant_balanced(seed=s)
    num_true = sum(f.truth_table.values()) # type: ignore
    print(s, num_true)  # must be 0, 8, or 16


0 16
1 8
2 8
3 8
4 8


## Problem 2: Classical Testing for Function Type

Deutsch's algorithm is designed to demonstrate a potential advantage of quantum computing over classical computation. To understand this advantage, we must first understand the classical cost of solving the underlying problem. Write a Python function determine_constant_balanced that takes as input a function f, as defined in Problem 1. The function should analyze f and return the string "constant" or "balanced" depending on whether the function is constant or balanced. Write a brief note on the efficiency of your solution. What is the maximum number of times you need to call f to be 100% certain whether it is constant or balanced?

Given a promised function `f(a, b, c, d)` that is either **constant** or **balanced**, we want to determine which it is.

A deterministic classical strategy is:

- Evaluate `f` on different inputs until either:
  - we see **both** `True` and `False` outputs → the function must be **balanced**, or
  - we have seen the **same output on 9 distinct inputs** → the function must be **constant**.

Why 9? With 4 input bits there are 16 inputs total. A balanced function has exactly 8 of each output.  
So after observing the same output on 8 inputs, it *could still* be balanced (we might have only sampled the “all False half”, for example).  
The **9th** matching output makes balanced impossible, so the function must be constant.

**Worst-case number of calls (100% certainty):** \(2^{n-1} + 1\). For \(n=4\): \(2^3 + 1 = 9\).

In [3]:
from itertools import product

def determine_constant_balanced(f) -> str:
    """
    Determine whether a promised constant/balanced 4-input Boolean function is
    "constant" or "balanced".

    Worst-case calls to f: 9 (guarantees 100% certainty under the promise).
    """
    first = None

    for i, x in enumerate(product([False, True], repeat=4), start=1):
        y = f(*x)

        if first is None:
            first = y
        elif y != first:
            return "balanced"

        # After 9 identical outputs, the function cannot be balanced (only 8 of each)
        if i == 9:
            return "constant"

    # With the promise, execution should always return before this.
    raise RuntimeError("Promise violated: function is neither constant nor balanced.")


In [ ]:
# Quick demo using Problem 1 generator
for s in range(6):
    f = random_constant_balanced(seed=s)
    print(f"seed={s:2d}  trues={sum(f.truth_table.values()):2d}  classified={determine_constant_balanced(f)}") # type: ignore

seed= 0  trues=16  classified=constant
seed= 1  trues= 8  classified=balanced
seed= 2  trues= 8  classified=balanced
seed= 3  trues= 8  classified=balanced
seed= 4  trues= 8  classified=balanced
seed= 5  trues=16  classified=constant


## Problem 3: Quantum Oracles

Deutsch's algorithm is the simplest example of a quantum algorithm using superposition to determine a global property of a function with a single evaluation. In the single-input case, there are four possible Boolean functions. Using Qiskit, create the appropriate quantum oracles for each of the possible single-Boolean-input functions used in Deutsch's algorithm. Demonstrate their use and explain how each oracle implements its corresponding function.

In the single-input case, there are four possible Boolean functions:

1. f(x) = 0        (constant)
2. f(x) = 1        (constant)
3. f(x) = x        (balanced)
4. f(x) = ¬x       (balanced)

The oracle must implement:

    U_f |x⟩|y⟩ = |x⟩ |y ⊕ f(x)⟩

This means the second qubit is flipped if and only if f(x) = 1.

Each oracle implements the transformation:

    |x⟩|y⟩ → |x⟩|y ⊕ f(x)⟩

- The constant functions either do nothing (f(x)=0) or always apply an X gate (f(x)=1).
- The balanced function f(x)=x is implemented using a CNOT.
- The balanced function f(x)=¬x is implemented by flipping the target first and then applying CNOT.

This matches the required quantum oracle definition.

In [5]:
from qiskit import QuantumCircuit

def deutsch_oracle(name: str) -> QuantumCircuit:
    """
    Construct a 2-qubit oracle for Deutsch's algorithm.

    Qubit 0: input |x⟩
    Qubit 1: output |y⟩ (target qubit)
    """
    qc = QuantumCircuit(2, name=f"U_{name}")

    if name == "f0":
        # f(x) = 0  → do nothing
        pass

    elif name == "f1":
        # f(x) = 1  → always flip y
        qc.x(1)

    elif name == "fx":
        # f(x) = x  → flip y if x = 1
        qc.cx(0, 1)

    elif name == "fnotx":
        # f(x) = ¬x → effectively y ⊕ (1 ⊕ x)
        qc.x(1)
        qc.cx(0, 1)

    else:
        raise ValueError("Invalid oracle name: use 'f0', 'f1', 'fx', or 'fnotx'.")

    return qc


In [6]:
for name in ["f0", "f1", "fx", "fnotx"]:
    oracle = deutsch_oracle(name)
    print(f"\nOracle: {name}")
    print(oracle.draw())  # ASCII diagram



Oracle: f0
     
q_0: 
     
q_1: 
     

Oracle: f1
          
q_0: ─────
     ┌───┐
q_1: ┤ X ├
     └───┘

Oracle: fx
          
q_0: ──■──
     ┌─┴─┐
q_1: ┤ X ├
     └───┘

Oracle: fnotx
               
q_0: ───────■──
     ┌───┐┌─┴─┐
q_1: ┤ X ├┤ X ├
     └───┘└───┘


In [7]:
from qiskit.visualization import circuit_drawer

for name in ["f0", "f1", "fx", "fnotx"]:
    oracle = deutsch_oracle(name)
    print(f"\nOracle: {name}")
    circuit_drawer(oracle, output="mpl")



Oracle: f0

Oracle: f1

Oracle: fx

Oracle: fnotx


Each oracle implements the transformation:

    |x⟩|y⟩ → |x⟩|y ⊕ f(x)⟩

- The constant functions either do nothing (f(x)=0) or always apply an X gate (f(x)=1).
- The balanced function f(x)=x is implemented using a CNOT.
- The balanced function f(x)=¬x is implemented by flipping the target first and then applying CNOT.

This matches the required quantum oracle definition.
